# 智能建造数据核验
只读分析；岗位ID、关系条数、招聘人数分开统计。企业完整名称列扫描结果见 enterprise-evidence.json，复现脚本为 audit.py。

In [ ]:
from pathlib import Path
import json, collections, sqlite3
ROOT=Path('/Users/liuhongzhe/Desktop/学堂/专业建设/Codex工程')
p=ROOT/'outputs/job-industry-stage-mapping-20260825'
jobs=json.loads((p/'jobs.json').read_text())
nodes=json.loads((p/'industry_nodes.json').read_text())
rels=json.loads((p/'relations.json').read_text())
cn=[n for n in nodes if n['chain_name']=='基础设施与城市建设产业链']
cr=[r for r in rels if r['chain_name']=='基础设施与城市建设产业链']
print(len(jobs),len(nodes),len(cn),len(cr),len({r['job_id'] for r in cr}))
print(collections.Counter(r['review_status'] for r in cr))
ids={'30010','30065','35689','63467','65311','66883','77866','78599'}
for j in jobs:
 if j['id'] in ids:
  print(j['id'],j['cleaned_position'],[(r['industry_node_id'],r['review_status']) for r in rels if r['job_id']==j['id']])

In [ ]:
import openpyxl
w=openpyxl.load_workbook(ROOT/'outputs/typical-task-full-20260902/岗位典型工作任务与原子能力项_参考文件与总览.xlsx',read_only=True,data_only=True)
for i,row in enumerate(w['岗位总览'].values,1):
 if str(row[0]) in ids: print(i,row)
w.close()
w=openpyxl.load_workbook(ROOT/'outputs/01a056a7-547c-7590-be55-84c913d7762c/产业环节上下游及桑基图关系.xlsx',read_only=True,data_only=True)
for i,row in enumerate(w.active.values,1):
 if i in {337,359,366}: print(i,row)
w.close()
print('10年复利期末倍数',1.121**10)

In [ ]:
c=sqlite3.connect(':memory:')
c.execute('create table job_node_relations(chain_name text,review_status text)')
c.executemany('insert into job_node_relations values (?,?)',[(x['chain_name'],x['review_status']) for x in rels])
print(c.execute("SELECT review_status AS status, COUNT(*) AS count FROM job_node_relations WHERE chain_name = '基础设施与城市建设产业链' GROUP BY review_status ORDER BY count DESC").fetchall())